In [5]:
import numpy as np
import pandas as pd
import os

In [6]:
path = r"C:\Datathon\Datathon2026_DataFusion4\data\raw"

customers         = pd.read_csv(os.path.join(path, "customers.csv"))
geography         = pd.read_csv(os.path.join(path, "geography.csv"))
inventory         = pd.read_csv(os.path.join(path, "inventory.csv"))
order_items       = pd.read_csv(os.path.join(path, "order_items.csv"))
orders            = pd.read_csv(os.path.join(path, "orders.csv"))
payments          = pd.read_csv(os.path.join(path, "payments.csv"))
products          = pd.read_csv(os.path.join(path, "products.csv"))
promotions        = pd.read_csv(os.path.join(path, "promotions.csv"))
returns           = pd.read_csv(os.path.join(path, "returns.csv"))
reviews           = pd.read_csv(os.path.join(path, "reviews.csv"))
sales             = pd.read_csv(os.path.join(path, "sales.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))
shipments         = pd.read_csv(os.path.join(path, "shipments.csv"))
web_traffic       = pd.read_csv(os.path.join(path, "web_traffic.csv"))

print("Load xong tất cả DataFrame")

C:\Users\Admin\AppData\Local\Temp\ipykernel_25756\2127364110.py:6: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items       = pd.read_csv(os.path.join(path, "order_items.csv"))


Load xong tất cả DataFrame


# CHECK CHẤT LƯỢNG DATA

In [7]:
def quick_explore(df, name):
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape}")
    print(f"\n--- Null counts ---")
    null_counts = df.isnull().sum()
    print(null_counts[null_counts > 0] if null_counts.any() else "Không có null")
    print(f"\n--- Dtypes ---")
    print(df.dtypes)
    print(f"\n--- Sample ---")
    print(df.head(3))

# Chạy cho tất cả bảng
tables = {
    "customers": customers,
    "geography": geography,
    "inventory": inventory,
    "order_items": order_items,
    "orders": orders,
    "payments": payments,
    "products": products,
    "promotions": promotions,
    "returns": returns,
    "reviews": reviews,
    "sales": sales,
    "shipments": shipments,
    "web_traffic": web_traffic,
}

for name, df in tables.items():
    quick_explore(df, name)


customers
Shape: (121930, 7)

--- Null counts ---
Không có null

--- Dtypes ---
customer_id             int64
zip                     int64
city                   object
signup_date            object
gender                 object
age_group              object
acquisition_channel    object
dtype: object

--- Sample ---
   customer_id    zip       city signup_date  gender age_group  \
0            1  15201  Hai Phong  2021-12-30  Female     35-44   
1            2  15201  Hai Phong  2013-12-27  Female     45-54   
2            3  15201  Hai Phong  2018-07-24  Female     18-24   

  acquisition_channel  
0        social_media  
1      email_campaign  
2      organic_search  

geography
Shape: (39948, 4)

--- Null counts ---
Không có null

--- Dtypes ---
zip          int64
city        object
region      object
district    object
dtype: object

--- Sample ---
     zip       city region      district
0  15201  Hai Phong   East  District #13
1  15202     Phu Ly   East  District #13
2  15203 

## Clean Data

In [8]:
#  Chuyển đổi các cột date
date_cols_map = {
    "customers":   ["signup_date"],
    "orders":      ["order_date"],
    "promotions":  ["start_date", "end_date"],
    "returns":     ["return_date"],
    "reviews":     ["review_date"],
    "sales":       ["Date"],
    "shipments":   ["ship_date", "delivery_date"],
    "inventory":   ["snapshot_date"],
    "web_traffic": ["date"],
}

for table_name, cols in date_cols_map.items():
    for col in cols:
        tables[table_name][col] = pd.to_datetime(tables[table_name][col])

print("Đã xong")

Đã xong


In [9]:
#  Zip -> String
for table_name in ["customers", "geography", "orders"]:
    tables[table_name]["zip"] = (
        tables[table_name]["zip"]
        .astype(str)
        .str.zfill(5)
    )

# Xử lý dữ liệu thiếu
# order_items: null promo => không có KM
order_items["promo_id"]   = order_items["promo_id"].fillna("NoPromo")
order_items["promo_id_2"] = order_items["promo_id_2"].fillna("NoPromo")

# promotions: null category = áp dụng tất cả
promotions["applicable_category"] = promotions["applicable_category"].fillna("All")

print("Xử lý xong")

Xử lý xong


In [10]:
# Check quy luật
# cogs < price (Giá vốn < giá bán)
price_violations = products[products["cogs"] >= products["price"]]
if len(price_violations) > 0:
    print(f"\nCảnh báo - Price violations: Có {len(price_violations)} dòng lỗi")
else:
    print(f"\nThành công - Price violations: Không có dòng lỗi")

# ship_date <= delivery_date (Ngày giao phải sau ngày ship)
delivery_violations = shipments[
    shipments["delivery_date"] < shipments["ship_date"]
]
if len(delivery_violations) > 0:
    print(f"Cảnh báo - Delivery trước ship: Có {len(delivery_violations)} dòng lỗi")
else:
    print(f"Thành công - Delivery trước ship: Không có dòng lỗi")

# order_date <= ship_date (Ngày giao sau ngày đặt hàng)
orders_ship = orders.merge(shipments, on="order_id", how="inner")
ship_violations = orders_ship[
    orders_ship["ship_date"] < orders_ship["order_date"]
]
if len(ship_violations) > 0:
    print(f"Cảnh báo - Ship trước order: Có {len(ship_violations)} dòng lỗi")
else:
    print("Thành công - Ship trước order: Không có dòng lỗi")

# rating 1-5 (Đánh giá luôn nằm trong ngưỡng 1-5)
rating_violations = reviews[~reviews["rating"].between(1, 5)]
if len(rating_violations) > 0:
    print(f"Cảnh báo- Rating ngoài khoảng 1-5: Có {len(rating_violations)} dòng lỗi")
else:
    print("Thành công - Rating hợp lệ: Không có dòng lỗi")


Thành công - Price violations: Không có dòng lỗi
Thành công - Delivery trước ship: Không có dòng lỗi
Thành công - Ship trước order: Không có dòng lỗi
Thành công - Rating hợp lệ: Không có dòng lỗi


## Tạo file 

In [11]:
# SALES MASTER 
# mỗi dòng = 1 sản phẩm trong 1 đơn hàng

# order_items + products
sales_master = order_items.merge(
    products[["product_id", "product_name", "category", 
              "segment", "size", "color", "price", "cogs"]],
    on="product_id",
    how="left"
)

# Thêm orders
sales_master = sales_master.merge(
    orders[["order_id", "order_date", "customer_id", 
            "order_status", "payment_method", 
            "device_type", "order_source", "zip"]],
    on="order_id",
    how="left"
)

# Thêm promotions (lấy thông tin KM)
sales_master = sales_master.merge(
    promotions[["promo_id", "promo_type", "discount_value",
                "promo_channel", "stackable_flag",
                "applicable_category", "min_order_value"]],
    on="promo_id",
    how="left"
)

print(f"Shape sau merge: {sales_master.shape}")

Shape sau merge: (714669, 27)


In [12]:
sales_master["promo_type"]          = sales_master["promo_type"].fillna("NoPromo")
sales_master["discount_value"]      = sales_master["discount_value"].fillna(0)
sales_master["promo_channel"]       = sales_master["promo_channel"].fillna("NoPromo")
sales_master["stackable_flag"]      = sales_master["stackable_flag"].fillna(0)
sales_master["applicable_category"] = sales_master["applicable_category"].fillna("NoPromo")
sales_master["min_order_value"]     = sales_master["min_order_value"].fillna(0)

In [13]:
from lunardate import LunarDate

tet_map = {}
for year in range(2012, 2025):
    tet_map[year] = pd.Timestamp(
        LunarDate(year, 1, 1).toSolarDate()
    )

def add_time_features(df, date_col="order_date"):
    df = df.copy()
    dt = df[date_col]

    df["year"]       = dt.dt.year
    df["month"]      = dt.dt.month
    df["day"]        = dt.dt.day
    df["dayofweek"]  = dt.dt.dayofweek
    df["quarter"]    = dt.dt.quarter
    df["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)

    # Tết features
    df["tet_date"]    = df["year"].map(tet_map)
    df["days_to_tet"] = (dt - df["tet_date"]).dt.days
    df["is_pre_tet"]  = df["days_to_tet"].between(-20, -1).astype(int)
    df["is_tet_week"] = df["days_to_tet"].between(0, 7).astype(int)
    df["is_post_tet"] = df["days_to_tet"].between(8, 22).astype(int)
    df = df.drop(columns=["tet_date"])

    # Mega Sale
    mega_sale_days = {(9,9), (10,10), (11,11), (12,12)}
    df["is_mega_sale"] = df.apply(
        lambda r: 1 if (r["month"], r["day"]) in mega_sale_days else 0,
        axis=1
    )

    return df

sales_master = add_time_features(sales_master)
print(f"\nFinal shape: {sales_master.shape}")


Final shape: (714669, 38)


In [14]:
# CUSTOMER MASTER 
# mỗi dòng = 1 khách hàng
# customers + geography
# Lấy thông tin vùng miền của từng khách hàng
customer_master = customers.merge(
    geography[["zip", "region", "district"]],
    on="zip",
    how="left"
)

# Aggregate orders theo customer
customer_orders = orders.groupby("customer_id").agg(
    total_orders     = ("order_id", "count"),
    first_order_date = ("order_date", "min"),
    last_order_date  = ("order_date", "max"),
    fav_device       = ("device_type", lambda x: x.mode()[0]),
    fav_order_source = ("order_source", lambda x: x.mode()[0]),
    fav_payment      = ("payment_method", lambda x: x.mode()[0]),
).reset_index()

# Aggregate revenue theo customer
# Chỉ tính đơn delivered — loại cancelled
delivered_orders = sales_master[
    sales_master["order_status"] == "delivered"
]

# Join tất cả lại
customer_master = customer_master.merge(
    customer_orders, on="customer_id", how="left"
)
print(f"Final merge xong: {customer_master.shape}")

Final merge xong: (121930, 15)


In [15]:
customer_master.isnull().sum()

customer_id                0
zip                        0
city                       0
signup_date                0
gender                     0
age_group                  0
acquisition_channel        0
region                     0
district                   0
total_orders           31684
first_order_date       31684
last_order_date        31684
fav_device             31684
fav_order_source       31684
fav_payment            31684
dtype: int64

In [16]:
# OPERATIONS MASTER 
# Grain: mỗi dòng = 1 đơn hàng
# Aggregate returns theo order
# returns có nhiều dòng/order → phải aggregate TRƯỚC khi join
returns_agg = returns.groupby("order_id").agg(
    total_return_qty  = ("return_quantity", "sum"),
    total_refund      = ("refund_amount", "sum"),
    return_count      = ("return_id", "count"),
    main_return_reason= ("return_reason", lambda x: x.mode()[0]),
).reset_index()


#  Aggregate reviews theo order
# reviews có nhiều dòng/order → phải aggregate TRƯỚC khi join
reviews_agg = reviews.groupby("order_id").agg(
    avg_rating  = ("rating", "mean"),
    review_count= ("review_id", "count"),
).reset_index()


#  join tất cả
ops_master = (
    orders
    .merge(shipments,   on="order_id", how="left")
    .merge(returns_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(geography[["zip", "region", "district"]],
           on="zip", how="left")
)

print(f"Final merge xong: {ops_master.shape}")

Final merge xong: (646945, 19)


In [17]:
ops_master.isnull().sum()

order_id                   0
order_date                 0
customer_id                0
zip                        0
order_status               0
payment_method             0
device_type                0
order_source               0
ship_date              80878
delivery_date          80878
shipping_fee           80878
total_return_qty      610883
total_refund          610883
return_count          610883
main_return_reason    610883
avg_rating            535576
review_count          535576
region                     0
district                   0
dtype: int64

In [18]:
# Xuất file
# Xuất file CSV
sales_master.to_csv("sales_master.csv", index=False, encoding="utf-8-sig")
customer_master.to_csv("customer_master.csv", index=False, encoding="utf-8-sig")
ops_master.to_csv("ops_master.csv", index=False, encoding="utf-8-sig")
customers.to_csv("customers.csv", index=False, encoding="utf-8-sig")
geography.to_csv("geography.csv", index=False, encoding="utf-8-sig")
inventory.to_csv("inventory.csv", index=False, encoding="utf-8-sig")
order_items.to_csv("order_items.csv", index=False, encoding="utf-8-sig")
orders.to_csv("orders.csv", index=False, encoding="utf-8-sig")
payments.to_csv("payments.csv", index=False, encoding="utf-8-sig")
products.to_csv("products.csv", index=False, encoding="utf-8-sig")
promotions.to_csv("promotions.csv", index=False, encoding="utf-8-sig")
returns.to_csv("returns.csv", index=False, encoding="utf-8-sig")
reviews.to_csv("reviews.csv", index=False, encoding="utf-8-sig")
sales.to_csv("sales.csv", index=False, encoding="utf-8-sig")
sample_submission.to_csv("sample_submission.csv", index=False, encoding="utf-8-sig")
shipments.to_csv("shipments.csv", index=False, encoding="utf-8-sig")
web_traffic.to_csv("web_traffic.csv", index=False, encoding="utf-8-sig")

print("Đã xuất file CSV thành công!")

Đã xuất file CSV thành công!
